# Lab 03 — Representation: the STFT vs the global spectrum

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 3 — §3.9–§3.10 (STFT/spectrogram and the uncertainty trade-off), §3.7 (resolution vs zero-padding).

**Biomedical question.** Is one global spectrum enough, or does the signal's content change over time — and *when*?
**Task type (§1.8).** Representation (time–frequency), not detection: the goal is to show the structure, not to threshold it.
**Information that must be preserved.** WHEN each rhythm occurs — the alpha→spindle ordering and the instant of the one-off transient — not merely *whether* they are present.
**Main assumptions.** the signal is non-stationary, but within a short analysis window it is approximately stationary (the premise that makes an STFT meaningful).
**Primary diagnostic.** put the global `rfft` spectrum next to the STFT/spectrogram, then sweep the window length to expose the time–frequency uncertainty trade.
**Transfer challenge.** pick an STFT window for an ECG in which a brief arrhythmic beat must be *timed* without losing the frequency resolution that separates two nearby rhythms.

*Self-contained: a seeded synthetic non-stationary EEG (no data files, no network), so it runs fully offline in a browser.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab03_representation_stft_vs_global/lab03_representation_stft_vs_global.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab03_representation_stft_vs_global.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab03_representation_stft_vs_global.ipynb)

In [ ]:
# --- shared setup (reproducible; self-contained synthetic non-stationary EEG) ---
import numpy as np, matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

FS, SECS = 100, 30
ALPHA_F, SPINDLE_F = 10.0, 13.0      # first-half rhythm -> second-half rhythm
SWITCH_T = 15.0                       # the alpha->spindle crossover (mid-record)
BLIP_T   = 22.0                       # the one-off K-complex-like transient (KNOWN instant)

def synth_eeg(fs=FS, secs=SECS):
    """Non-stationary EEG-like signal: a ~10 Hz alpha in the first half that hands off to a
    ~13 Hz spindle in the second half, one brief broadband transient at BLIP_T, plus
    broadband noise. The time-structure is REAL: which rhythm dominates depends on WHEN."""
    t = np.arange(int(fs*secs))/fs
    w2 = 1.0/(1.0 + np.exp(-(t - SWITCH_T)/0.5))    # smooth 0->1 crossfade at SWITCH_T
    w1 = 1.0 - w2
    x  = w1*np.sin(2*np.pi*ALPHA_F*t)               # alpha, first half
    x += w2*np.sin(2*np.pi*SPINDLE_F*t)             # spindle, second half
    x += 4.0*np.exp(-((t - BLIP_T)/0.01)**2)        # sharp broadband transient (K-complex-like)
    x += 0.30*rng.standard_normal(t.size)           # broadband background noise
    return t, x

t, x = synth_eeg()
fs = FS
print(f"signal: {x.size} samples, fs={fs} Hz, {x.size/fs:.0f} s; "
      f"alpha {ALPHA_F} Hz -> spindle {SPINDLE_F} Hz at {SWITCH_T} s; transient at {BLIP_T} s")

## 1. See it — the raw time series
Plot the whole record and zoom on the transient. Name what you can (and cannot) read off the raw trace.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 2.8), gridspec_kw={"width_ratios": [3, 1]})
ax[0].plot(t, x, lw=0.6); ax[0].axvline(BLIP_T, color="r", ls="--", lw=1)
ax[0].set_xlabel("s"); ax[0].set_title("Non-stationary EEG (full 30 s)")
zoom = (t > BLIP_T - 0.6) & (t < BLIP_T + 0.6)
ax[1].plot(t[zoom], x[zoom], lw=0.8); ax[1].axvline(BLIP_T, color="r", ls="--", lw=1)
ax[1].set_xlabel("s"); ax[1].set_title(f"transient @ {BLIP_T:.0f}s")
plt.tight_layout(); plt.show()
# Checkpoint: you can SEE a big blip near 22 s, and the wave looks a touch faster late in the
# record -- but can you read the exact rhythms, or WHEN alpha gives way to spindle, from the
# raw trace alone? That is what the representation choices below have to recover.

## 2. The global spectrum — present, but timeless
`# TODO` compute the magnitude spectrum of the **whole** record with `np.fft.rfft` and plot it. This is the single-picture summary you get if you assume the signal is stationary.

In [ ]:
# TODO compute the GLOBAL magnitude spectrum of the whole record with np.fft.rfft and plot it
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: two lines near 10 and 13 Hz -- so both rhythms exist. But WHICH came first? And
# the one-off transient: its energy is smeared thinly across ALL frequencies, so it barely
# dents this curve. A single global spectrum answers "present?" but never "WHEN?".

## 3. The STFT/spectrogram — recover WHEN
`# TODO` compute the spectrogram with `scipy.signal.spectrogram` and plot it in dB. A moving window turns one timeless picture into frequency-vs-time, so the ordering becomes visible.

In [ ]:
# TODO compute the spectrogram (scipy.signal.spectrogram, nperseg=256) and plot 10*log10 in dB
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: now WHEN is visible -- alpha fills the first half, the spindle takes over the
# second, and the transient shows up as a vertical broadband stripe at its instant. The
# global spectrum of Cell 2 could show neither the ordering nor the transient's time.

## 4. The time–frequency uncertainty trade — you can't win both
`# TODO` sweep the window length `nperseg`. A short window pins events in **time** but coarsens **frequency** (bin width `df = fs/nperseg`); a long window sharpens frequency but spreads each event over its **duration** (`dt = nperseg/fs`). Measure both ends of the trade and print a small table.

In [ ]:
# TODO sweep the STFT window length and MEASURE the time-frequency trade-off. For each
#   nperseg report: window duration dt=nperseg/fs (time resolution), bin width df=fs/nperseg
#   (frequency resolution), whether df resolves the 3 Hz alpha/spindle gap, the measured
#   first/second-half peak Hz, and the measured temporal width (FWHM) of the transient stripe.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: short window -> tiny dt (blip pinned in time) but coarse df (10 & 13 Hz collapse
# toward one bin, so peak1~peak2 and the alpha->spindle shift hides); long window -> fine df
# (10 vs 13 cleanly split) but large dt (the blip's energy smears across seconds). No single
# window minimizes both -- that is the uncertainty principle, not a coding bug.

## 5. Live sanity check — the two halves peak at different rhythms
A representation you never test is a decoration. At a balanced window the *measured* dominant frequency must rise from ~10 Hz (first half) to ~13 Hz (second half): the alpha→spindle shift, computed, not asserted.

In [ ]:
# --- live sanity check: the dominant rhythm shifts up in the second half (numbers computed) ---
f_s, t_s, Sxx = sig.spectrogram(x, fs=fs, nperseg=256, noverlap=128, scaling="spectrum")
band = f_s <= 30
peak1 = f_s[band][np.argmax(Sxx[band][:, t_s <  SWITCH_T].mean(1))]
peak2 = f_s[band][np.argmax(Sxx[band][:, t_s >= SWITCH_T].mean(1))]
print(f"first-half  dominant peak = {peak1:.2f} Hz  (expect ~{ALPHA_F:.0f}, the alpha)")
print(f"second-half dominant peak = {peak2:.2f} Hz  (expect ~{SPINDLE_F:.0f}, the spindle)")
assert 8.5 <= peak1 <= 11.5 and 11.5 <= peak2 <= 14.5 and peak2 > peak1, \
    "the alpha->spindle shift should read as a higher second-half peak"
print("OK: the dominant rhythm moves up in the second half -- WHEN is a real, measurable fact.")

## Reflection
1. **Stable vs changed.** Which conclusion held across the *longer* window lengths you tried, and which one depended on the window? (Hint: "both ~10 and ~13 Hz rhythms are present" holds once the window is long enough to resolve them — the shortest window you tried can still merge them into one peak — vs "which one comes first / when the transient occurs.")
2. **Evidence for a real recording.** Before trusting an alpha→spindle transition *time* on a real overnight EEG, what would you need to check — sampling rate, montage/reference, artifact rejection, and whether the shift survives more than one window length?
3. **Rule out.** Name a representation choice that is *wrong* for this spec and the requirement it breaks.

> *Your answers here.*